In [1]:
!pip install pandas

In [196]:
import pandas as pd
import sqlite3
from sqlite3 import Error

## Подключение к бд

In [197]:
def create_connection(path):
    connection = None
    try:
        connection = sqlite3.connect(path)
    except Error as e:
        print(f"The error {e} occured")
    return connection

connection = create_connection("../checking-logs.sqlite")

In [198]:
pd.read_sql("PRAGMA table_info(test)", connection)

,cid,name,type,notnull,dflt_value,pk
0,0,index,INTEGER,0,None,0
1,1,uid,TEXT,0,None,0
2,2,labname,TEXT,0,None,0
3,3,first_commit_ts,TIMESTAMP,0,None,0
4,4,first_view_ts,TIMESTAMP,0,None,0


In [199]:
pd.read_sql("SELECT * FROM test LIMIT 10;", connection)

,index,uid,labname,first_commit_ts,first_view_ts
0,0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
5,5,user_1,project1,2020-05-14 20:56:08.898880,2020-04-26 21:53:59.624136
6,6,user_10,laba04,2020-04-25 08:24:52.696624,2020-04-18 12:19:50.182714
7,7,user_10,laba04s,2020-04-25 08:37:54.604222,2020-04-18 12:19:50.182714
8,8,user_10,laba05,2020-05-01 19:27:26.063245,2020-04-18 12:19:50.182714
9,9,user_10,laba06,2020-05-19 11:39:28.885637,2020-04-18 12:19:50.182714


In [6]:
pd.read_sql("SELECT * FROM deadlines", connection)

,index,labs,deadlines
0,0,laba04,1587945599
1,1,laba04s,1587945599
2,2,laba05,1588550399
3,4,laba06,1590364799
4,5,laba06s,1590364799
5,3,project1,1589673599


## Минимальная разница между первым коммитом и дедлайном 

In [176]:
query = "SELECT  uid, (MIN(strftime('%s', first_commit_ts) - deadlines.deadlines)) / 3600 AS difference  " \
"FROM test LEFT JOIN deadlines ON labs = labname " \
"WHERE labname <> 'project1' " \
"GROUP BY uid" \

df_min = pd.read_sql(query, connection)

In [116]:
df_min

,uid,difference
0,user_1,-175
1,user_10,-132
2,user_14,-200
3,user_17,-81
4,user_18,-10
5,user_19,-148
6,user_21,-126
7,user_25,-150
8,user_28,-174
9,user_3,-182


## Максимальная разница между первым коммитом и дедлайном

In [177]:
query = "SELECT  uid, (MAX(strftime('%s', first_commit_ts) - deadlines.deadlines)) / 3600 AS difference  " \
"FROM test LEFT JOIN deadlines ON labs = labname " \
"WHERE labname <> 'project1' " \
"GROUP BY uid" \

df_max = pd.read_sql(query, connection)

In [118]:
df_max

,uid,difference
0,user_1,-6
1,user_10,-39
2,user_14,-84
3,user_17,-34
4,user_18,-3
5,user_19,-32
6,user_21,-33
7,user_25,-2
8,user_28,-8
9,user_3,-60


## Средняя дельта между первым коммитом и дедлайном

In [ ]:
query = "SELECT (AVG(strftime('%s', first_commit_ts) - deadlines.deadlines)) / 3600.00 AS difference  " \
"FROM test LEFT JOIN deadlines ON labs = labname " \
"WHERE labname <> 'project1'" \


df_avg = pd.read_sql(query, connection)

In [124]:
df_avg

,difference
0,-89.687841


## Подтаблица, удовлетворяющая следующим требованиям:
* создайте таблицу со столбцами: uid, avg_diff, pageviews
* uid - это uids, которые существуют в тесте
* avg_diff - это средняя дельта между первым фиксацией и крайним сроком лабораторной работы для пользователя
* pageviews - это количество посещений новостной ленты на одного пользователя
* не учитывайте лабораторный проект «project1»
* сохранить его в dataframe views_diff

In [188]:
query = """
SELECT uid, 
    (AVG(strftime('%s', first_commit_ts) - deadlines.deadlines)) / 3600.00 AS avg_diff,
    (SELECT COUNT(*) FROM pageviews WHERE test.uid = pageviews.uid) AS pageviews
FROM test LEFT JOIN deadlines ON labs = labname
WHERE labname <> 'project1' AND pageviews > 1
GROUP BY uid;
"""

views_diff = pd.read_sql(query, connection)

In [157]:
views_diff

,uid,avg_diff,pageviews
0,user_1,-65.119778,28
1,user_10,-75.242444,89
2,user_14,-159.568796,143
3,user_17,-62.207667,47
4,user_18,-6.368148,3
5,user_19,-99.440417,16
6,user_21,-96.111181,10
7,user_25,-93.474944,179
8,user_28,-86.793833,149
9,user_3,-105.738222,317


In [193]:
views_diff.drop(9, inplace = True)

## Расчет коэффициента корреляции

In [194]:
views_diff[["pageviews", "avg_diff"]].corr(method="spearman")

,pageviews,avg_diff
pageviews,1.00000,-0.06687
avg_diff,-0.06687,1.00000


## Закрытие соединения

In [195]:
connection.close()